# Track 01 — LLM SFT đầy đủ: data → train → validate → biểu đồ → inference

Nhiệm vụ: dạy `Qwen3-0.6B` dịch **câu hỏi tiếng Anh + schema → câu SQL**.

**Kết quả đã chạy thật** (RTX 5060 Ti, 4000 mẫu, 1 epoch, 108 giây):

| | Khớp chính xác / 60 mẫu held-out |
|---|---|
| Qwen3-0.6B gốc | **0 / 60 (0%)** |
| + LoRA r=16 | **44 / 60 (73.3%)** |

Model gốc không *thiếu kiến thức SQL* — nó không biết **phải xuất ra cái gì**. Đó đúng là loại
vấn đề fine-tune giải quyết được mà prompt engineering thì không (ở cỡ model 0.6B).

**Notebook này chạy được cả Kaggle T4 (fp16) và GPU Blackwell (bf16)** — tự nhận diện.

### Mục lục
1. Cài thư viện · 2. Kiểm tra môi trường · 3. Dữ liệu + **label masking** · 4. Phân tích độ dài (📊)
5. Collator · 6. LoRA · 7. Train + log · 8. **Biểu đồ loss** (📊) · 9. Validate + **biểu đồ so sánh** (📊)
10. Phân tích lỗi (📊) · 11. **Inference** · 12. Merge + xuất model · 13. Thí nghiệm

## 1. Cài thư viện

Ghim phiên bản đã verify 11/2026. `transformers` **v5** khác v4 nhiều.

In [ ]:
!pip install -q -U "transformers==5.22.1" "trl==1.17.1" "peft==0.21.2" "datasets==5.1.0" accelerate
# Kaggle cai san torchao 0.10.0 nhung peft 0.21.2 doi >=0.16.0 -> peft.is_torchao_available()
# RAISE ImportError ngay khi gan LoRA. Minh khong dung torchao nen go han la xong.
!pip uninstall -q -y torchao 2>/dev/null; echo "da go torchao"

## 2. Kiểm tra môi trường

⚠️ **T4 của Kaggle là sm_75 — KHÔNG hỗ trợ bf16.** Ô này tự chọn fp16/bf16.

In [ ]:
import torch, transformers, trl, peft, datasets, json, time, re, os
for m in (torch, transformers, trl, peft, datasets):
    print(f"{m.__name__:14s}", m.__version__)

assert torch.cuda.is_available(), "Bat GPU: Settings -> Accelerator -> GPU T4 x2"
cap = torch.cuda.get_device_capability(0)
print("\nGPU         :", torch.cuda.get_device_name(0))
print("compute cap :", cap)
print("VRAM        : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1e9))

# !! BAY DA GAP THAT: torch.cuda.is_bf16_supported() TRA VE True CA TREN T4 (sm_75),
# vi no tinh ca bf16 "emulation" - chay duoc nhung KHONG dung tensor core -> cham hon fp16 nhieu.
# Lan chay dau tien cua notebook nay da roi vao bay do: T4 cap (7,5) ma van chon bf16, 623 giay.
# Kiem tra DUNG la compute capability: bf16 native chi co tu sm_80 (Ampere) tro len.
print("torch.cuda.is_bf16_supported() bao :", torch.cuda.is_bf16_supported(), " <- KHONG tin duoc")
BF16  = cap[0] >= 8
DTYPE = torch.bfloat16 if BF16 else torch.float16
print("compute cap[0] >= 8                :", BF16, " <- day moi dung")
print(f"\n-> dung {'bf16' if BF16 else 'fp16'}  (T4=fp16, Ampere/Ada/Blackwell=bf16)")

In [ ]:
# ---- Style bieu do: palette da qua validator CVD (deltaE >= 8), light surface ----
import matplotlib as mpl, matplotlib.pyplot as plt

SURFACE="#fcfcfb"; INK="#0b0b0b"; INK2="#52514e"; GRID="#e5e4e0"
S1="#2a78d6"   # blue   - slot 1 (truoc / train)
S2="#eb6834"   # orange - slot 2 (sau / val)
S3="#1baf7a"   # aqua   - slot 3

mpl.rcParams.update({
    "figure.facecolor":SURFACE, "axes.facecolor":SURFACE,
    "axes.edgecolor":GRID, "axes.labelcolor":INK2, "text.color":INK,
    "xtick.color":INK2, "ytick.color":INK2,
    "axes.spines.top":False, "axes.spines.right":False,
    "grid.color":GRID, "grid.linewidth":0.8,
    "font.size":10, "axes.titlesize":12, "axes.titleweight":"600",
    "lines.linewidth":2, "lines.markersize":6,
    "figure.dpi":120,
})
def finish(ax, title, ylab=None, xlab=None, legend=False):
    ax.set_title(title, loc="left", pad=12, color=INK)
    if ylab: ax.set_ylabel(ylab)
    if xlab: ax.set_xlabel(xlab)
    ax.grid(axis="y", alpha=.7); ax.set_axisbelow(True)
    if legend: ax.legend(frameon=False, loc="best")
    plt.tight_layout()

print("style bieu do da nap")

## 3. Dữ liệu + **label masking** — phần cốt lõi nhất của SFT

Trong SFT chỉ phần **câu trả lời** được tính loss; phần prompt đặt label `-100`.

Không mask thì model bị phạt vì "không đoán được câu hỏi của người dùng" — việc vô nghĩa.
Loss vẫn giảm đẹp, bạn vẫn tưởng train tốt, nhưng chất lượng kém hơn rõ. **Lỗi im lặng phổ biến nhất.**

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

MODEL   = "Qwen/Qwen3-0.6B"
N_TRAIN, N_VAL, N_EVAL = 4000, 200, 60
MAX_LEN = 512
SYSTEM  = ("Ban la bo dich text-to-SQL. Dua vao schema va cau hoi, "
           "chi tra ve DUY NHAT cau truy van SQLite, khong giai thich.")

tok = AutoTokenizer.from_pretrained(MODEL)
raw = load_dataset("b-mc2/sql-create-context", split="train").shuffle(seed=42)
print("Tong:", len(raw), "| cot:", raw.column_names)

def build_prompt(ex):
    msgs=[{"role":"system","content":SYSTEM},
          {"role":"user","content":f"Schema:\n{ex['context']}\n\nCau hoi: {ex['question']}"}]
    # enable_thinking=False: Qwen3 co che do thinking, tat cho on dinh va ngan gon
    return tok.apply_chat_template(msgs, tokenize=False,
                                   add_generation_prompt=True, enable_thinking=False)

def to_features(ex):
    p = tok(build_prompt(ex), add_special_tokens=False)["input_ids"]
    a = tok(ex["answer"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    return {"input_ids":(p+a)[:MAX_LEN], "labels":([-100]*len(p)+a)[:MAX_LEN]}

ex = raw[0]
print("\n--- PROMPT (model KHONG hoc) ---"); print(build_prompt(ex))
print("--- ANSWER (model PHAI hoc) ---");    print(ex["answer"])
f = to_features(ex); nm = sum(1 for x in f["labels"] if x==-100)
print(f"\ntong {len(f['input_ids'])} tok | mask {nm} | tinh loss {len(f['labels'])-nm}")
print("labels[:8] :", f["labels"][:8],  "<- -100 = bo qua")
print("labels[-8:]:", f["labels"][-8:], "<- phan tinh loss")

In [ ]:
train = raw.select(range(N_TRAIN)).map(to_features, remove_columns=raw.column_names, desc="train")
val   = raw.select(range(N_TRAIN, N_TRAIN+N_VAL)).map(to_features, remove_columns=raw.column_names, desc="val")
ev    = raw.select(range(N_TRAIN+N_VAL, N_TRAIN+N_VAL+N_EVAL))   # held-out cho eval cuoi
print("train", len(train), "| val", len(val), "| eval", len(ev))
print("3 tap KHONG giao nhau -> khong leak")

## 4. 📊 Phân tích độ dài token — làm TRƯỚC khi train

Mẫu dài hơn `MAX_LEN` sẽ bị cắt → nhãn cụt → model học sinh ra câu cụt. Phải xem phân phối trước.

In [ ]:
import numpy as np
lens = np.array([len(x) for x in train["input_ids"]])
p50,p95,p99 = np.percentile(lens,[50,95,99])
n_trunc = int((lens>=MAX_LEN).sum())

fig, ax = plt.subplots(figsize=(7.2,3.4))
ax.hist(lens, bins=50, color=S1, edgecolor=SURFACE, linewidth=.5)
for v,lab in [(p50,f"p50={p50:.0f}"),(p95,f"p95={p95:.0f}"),(MAX_LEN,f"MAX_LEN={MAX_LEN}")]:
    ax.axvline(v, color=INK2, linestyle="--", linewidth=1)
    ax.annotate(lab, (v,ax.get_ylim()[1]*.92), color=INK2, fontsize=9,
                ha="right", rotation=90, va="top")
finish(ax, "Phan phoi do dai token (tap train)", "so mau", "token")
plt.show()

print(f"p50={p50:.0f}  p95={p95:.0f}  p99={p99:.0f}  max={lens.max()}")
print(f"Bi cat (>= MAX_LEN): {n_trunc}/{len(lens)}  ({100*n_trunc/len(lens):.2f}%)")
print("-> duoi 1% la an toan. Neu cao, tang MAX_LEN hoac loc bo mau dai.")

## 5. Collator tự viết

Viết tay một lần để hiểu: pad `input_ids` bằng `pad_token`, pad `labels` bằng **`-100`** (không phải pad_id), `attention_mask`=0 ở chỗ pad.

In [ ]:
from dataclasses import dataclass
@dataclass
class Collator:
    pad_id:int
    def __call__(self, feats):
        n=max(len(f["input_ids"]) for f in feats); ids=[];labs=[];att=[]
        for f in feats:
            k=n-len(f["input_ids"])
            ids.append(f["input_ids"]+[self.pad_id]*k)
            labs.append(f["labels"]  +[-100]*k)
            att.append([1]*len(f["input_ids"])+[0]*k)
        t=torch.tensor
        return {"input_ids":t(ids),"labels":t(labs),"attention_mask":t(att)}
coll=Collator(tok.pad_token_id)
print({k:tuple(v.shape) for k,v in coll([train[0],train[1]]).items()})

## 6. Gắn LoRA

`target_modules` phủ **hết 7 linear** — chỉ attention là không đủ để bằng full fine-tune.

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE, device_map="cuda")
model.config.use_cache=False
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
model.print_trainable_parameters()

## 7. Train — có đo val loss để phát hiện overfit

⚠️ `transformers` v5 **bỏ `warmup_ratio`** → dùng `warmup_steps`. Tutorial viết cho v4 sẽ lỗi `TypeError`.
Tra nhanh: `[p for p in inspect.signature(TrainingArguments.__init__).parameters if 'warm' in p]`

In [ ]:
from transformers import Trainer, TrainingArguments
OUT="/kaggle/working/qwen3-0.6b-sql-lora"

args = TrainingArguments(
    output_dir=OUT,
    per_device_train_batch_size=8, gradient_accumulation_steps=2,   # batch hieu dung 16
    num_train_epochs=1,
    learning_rate=2e-4,              # LoRA chiu lr cao hon full FT ~100x
    lr_scheduler_type="cosine", warmup_steps=8,
    logging_steps=10,
    eval_strategy="steps", eval_steps=25,      # <- de ve duoc duong val loss
    per_device_eval_batch_size=8,
    bf16=BF16, fp16=not BF16,
    save_strategy="no", report_to=[],
)
trainer = Trainer(model=model, args=args, train_dataset=train,
                  eval_dataset=val, data_collator=coll)
t0=time.time(); trainer.train(); train_secs=time.time()-t0
model.save_pretrained(OUT); tok.save_pretrained(OUT)
print(f"\nTrain xong trong {train_secs:.0f}s. Adapter: {OUT}")

## 8. 📊 Biểu đồ loss — train vs val

Đây là biểu đồ quan trọng nhất của một lần train. Cách đọc:
- **hai đường cùng giảm** → đang học tốt, có thể train thêm
- **train giảm, val đi ngang hoặc tăng** → **overfit**, phải dừng
- **cả hai đi ngang ở mức cao** → underfit: tăng lr, tăng `r`, hoặc thêm data

In [ ]:
H = trainer.state.log_history
tr = [(h["step"],h["loss"])      for h in H if "loss"      in h]
va = [(h["step"],h["eval_loss"]) for h in H if "eval_loss" in h]

fig, ax = plt.subplots(figsize=(7.2,3.8))
ax.plot(*zip(*tr), color=S1, label="train loss")
if va:
    ax.plot(*zip(*va), color=S2, label="val loss", marker="o")
    ax.annotate(f"{va[-1][1]:.3f}", va[-1], textcoords="offset points",
                xytext=(6,4), color=INK2, fontsize=9)
ax.annotate(f"{tr[-1][1]:.3f}", tr[-1], textcoords="offset points",
            xytext=(6,-10), color=INK2, fontsize=9)
finish(ax, "Loss theo step", "loss", "step", legend=True)
plt.show()

print(f"train loss dau -> cuoi : {tr[0][1]:.4f} -> {tr[-1][1]:.4f}")
if va: print(f"val   loss dau -> cuoi : {va[0][1]:.4f} -> {va[-1][1]:.4f}")
if va and va[-1][1] > min(v for _,v in va)*1.05:
    print("!! val loss da tang tro lai -> dau hieu OVERFIT, giam epoch")
else:
    print("OK: val loss chua tang tro lai")

## 9. Validate + 📊 biểu đồ so sánh trước/sau

**Đừng tin loss.** Loss giảm đẹp không có nghĩa model làm được việc. Phải đo trên dữ liệu
chưa từng thấy, bằng thước đo của chính nhiệm vụ: ở đây là **khớp chính xác câu SQL** sau chuẩn hoá.

In [ ]:
from peft import PeftModel
tok.padding_side="left"          # generate theo batch PHAI pad ben trai
prompts=[build_prompt(e) for e in ev]; golds=[e["answer"] for e in ev]
def norm(s): return re.sub(r"\s+"," ",s.strip().rstrip(";")).lower()

@torch.no_grad()
def run(m, label):
    m.eval(); outs=[]; t0=time.time()
    for i in range(0,len(prompts),12):
        bt=tok(prompts[i:i+12],return_tensors="pt",padding=True,
               add_special_tokens=False).to("cuda")
        g=m.generate(**bt,max_new_tokens=96,do_sample=False,pad_token_id=tok.pad_token_id)
        outs+=[tok.decode(g[j][bt["input_ids"].shape[1]:],skip_special_tokens=True).strip()
               for j in range(g.shape[0])]
    hit=sum(norm(o)==norm(g) for o,g in zip(outs,golds))
    dt=time.time()-t0
    print(f"{label:22s} {hit:2d}/{len(golds)}  ({100*hit/len(golds):5.1f}%)   {dt:.1f}s")
    return outs, hit, dt

base=AutoModelForCausalLM.from_pretrained(MODEL,dtype=DTYPE,device_map="cuda")
print("ket qua tren", len(ev), "mau CHUA TUNG THAY:")
o_base,h_base,t_base = run(base,"Qwen3-0.6B GOC")
ft = PeftModel.from_pretrained(base, OUT)
o_ft,  h_ft,  t_ft   = run(ft,  "Qwen3-0.6B + LoRA")

In [ ]:
acc_b, acc_f = 100*h_base/len(ev), 100*h_ft/len(ev)
fig, ax = plt.subplots(figsize=(5.6,3.6))
bars = ax.bar(["Goc","+ LoRA"], [acc_b,acc_f], width=.5,
              color=[S1,S2], edgecolor=SURFACE, linewidth=2)
for b,v,h in zip(bars,[acc_b,acc_f],[h_base,h_ft]):     # direct label = relief
    ax.annotate(f"{v:.1f}%\n{h}/{len(ev)}", (b.get_x()+b.get_width()/2, v),
                textcoords="offset points", xytext=(0,6),
                ha="center", color=INK, fontsize=10, fontweight="600")
ax.set_ylim(0,max(acc_f*1.28,12))
finish(ax, f"Khop chinh xac SQL - {len(ev)} mau held-out", "% khop")
plt.show()

# Bang so (table view - bat buoc kem theo bieu do)
print(f"{'Model':24s}{'Khop':>8s}{'%':>8s}{'Thoi gian':>11s}")
print("-"*51)
print(f"{'Qwen3-0.6B goc':24s}{h_base:>8d}{acc_b:>7.1f}%{t_base:>10.1f}s")
print(f"{'Qwen3-0.6B + LoRA':24s}{h_ft:>8d}{acc_f:>7.1f}%{t_ft:>10.1f}s")

## 10. 📊 Phân tích lỗi — model sai KIỂU gì?

Một con số accuracy không nói model sai ở đâu. Phân loại lỗi mới biết phải sửa gì.

In [ ]:
def classify(pred, gold):
    p,g = norm(pred), norm(gold)
    if p==g:                                   return "dung"
    if not p.startswith("select"):             return "khong ra SQL"
    if p.split()[:1]!=g.split()[:1]:           return "sai menh de dau"
    if ("where" in g) != ("where" in p):       return "thieu/thua WHERE"
    if ("count" in g) != ("count" in p):       return "sai ham tong hop"
    return "sai chi tiet"

from collections import Counter
cb = Counter(classify(p,g) for p,g in zip(o_base,golds))
cf = Counter(classify(p,g) for p,g in zip(o_ft,  golds))
keys = ["dung","sai chi tiet","thieu/thua WHERE","sai ham tong hop","sai menh de dau","khong ra SQL"]
keys = [k for k in keys if cb[k] or cf[k]]
y = np.arange(len(keys)); hh=.36

fig, ax = plt.subplots(figsize=(7.6, .62*len(keys)+1.8))
ax.barh(y+hh/2, [cb[k] for k in keys], height=hh, color=S1,
        label="Goc", edgecolor=SURFACE, linewidth=2)
ax.barh(y-hh/2, [cf[k] for k in keys], height=hh, color=S2,
        label="+ LoRA", edgecolor=SURFACE, linewidth=2)
ax.set_yticks(y); ax.set_yticklabels(keys); ax.invert_yaxis()
for i,k in enumerate(keys):
    for off,src,col in [(hh/2,cb,S1),(-hh/2,cf,S2)]:
        if src[k]: ax.annotate(str(src[k]),(src[k],i+off),textcoords="offset points",
                               xytext=(5,0),va="center",color=INK2,fontsize=9)
ax.grid(axis="x",alpha=.7); ax.set_axisbelow(True)
ax.set_title("Phan loai loi", loc="left", pad=12, color=INK)
ax.set_xlabel("so mau"); ax.legend(frameon=False, loc="lower right"); plt.tight_layout()
plt.show()

print(f"{'Kieu loi':22s}{'Goc':>6s}{'LoRA':>7s}")
print("-"*35)
for k in keys: print(f"{k:22s}{cb[k]:>6d}{cf[k]:>7d}")

## 10b. 📊 Multi-seed — con số σ **của accuracy**, thứ duy nhất dùng được

Đây là mục quan trọng nhất của notebook nếu bạn muốn công bố kết quả.

**Vấn đề:** ở trên ta đo được một con số accuracy. Lần sau đổi `r=32` ra số khác 2%.
Đó là cải thiện thật, hay chỉ là nhiễu do seed? Một lần chạy **không trả lời được**.

⚠️ Và phải đo σ của **accuracy**, không phải σ của train loss. σ của loss không cho biết gì
về việc chênh lệch accuracy có ý nghĩa hay không — hai đại lượng khác nhau.

Ô dưới train lại từ đầu với các seed khác, **giữ nguyên tập eval**, rồi tính `mean ± std` của accuracy.

In [ ]:
SEEDS = [1, 2]          # seed 0 da chay o tren. Them 2 seed -> 3 diem do.
                        # Moi seed ~10 phut tren T4. Dat [] de bo qua.
accs = [acc_f]          # ket qua seed mac dinh o tren

def train_and_eval(seed):
    torch.manual_seed(seed)
    m = get_peft_model(
        AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE, device_map="cuda"),
        LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                   task_type="CAUSAL_LM",
                   target_modules=["q_proj","k_proj","v_proj","o_proj",
                                   "gate_proj","up_proj","down_proj"]))
    m.config.use_cache = False
    a = TrainingArguments(
        output_dir=f"/kaggle/working/_seed{seed}",
        per_device_train_batch_size=8, gradient_accumulation_steps=2,
        num_train_epochs=1, learning_rate=2e-4,
        lr_scheduler_type="cosine", warmup_steps=8, logging_steps=100,
        bf16=BF16, fp16=not BF16, save_strategy="no", report_to=[],
        seed=seed, data_seed=seed)
    Trainer(model=m, args=a, train_dataset=train, data_collator=coll).train()
    _, hit, _ = run(m, f"seed {seed}")      # CUNG tap eval, CUNG thuoc do
    del m; gc_clear()
    return 100*hit/len(ev)

import gc as _gc
def gc_clear(): _gc.collect(); torch.cuda.empty_cache()

for sd in SEEDS:
    accs.append(train_and_eval(sd))

mu, sd_ = float(np.mean(accs)), float(np.std(accs))
print(f"\naccuracy qua {len(accs)} seed: " + ", ".join(f"{a:.2f}%" for a in accs))
print(f"=> {mu:.2f}% +/- {sd_:.2f}%  (do lech chuan)")
print(f"=> Moi 'cai thien' NHO HON ~{2*sd_:.2f}% la KHONG phan biet duoc voi nhieu.")
print("   Day moi la con so cho phep ban noi 'cau hinh A tot hon B'.")

In [ ]:
if len(accs) > 1:
    fig, ax = plt.subplots(figsize=(6.0,3.6))
    xs = [f"seed {i}" for i in range(len(accs))]
    ax.bar(xs, accs, width=.5, color=S1, edgecolor=SURFACE, linewidth=2)
    ax.axhline(mu, color=S2, linestyle="--", linewidth=2, label=f"trung binh {mu:.2f}%")
    ax.fill_between([-.5,len(accs)-.5], mu-sd_, mu+sd_, color=S2, alpha=.12,
                    label=f"+/- 1 sd ({sd_:.2f}%)")
    ax.set_xlim(-.5,len(accs)-.5); ax.set_ylim(0, max(accs)*1.2)
    for i,a in enumerate(accs):
        ax.annotate(f"{a:.1f}%", (i,a), textcoords="offset points", xytext=(0,5),
                    ha="center", color=INK, fontsize=9, fontweight="600")
    finish(ax, "Accuracy qua cac seed (cung config, cung tap eval)", "% khop", legend=True)
    plt.show()
    print(f"{'Seed':10s}{'Accuracy':>10s}")
    print("-"*20)
    for i,a in enumerate(accs): print(f"{i:<10d}{a:>9.2f}%")
else:
    print("SEEDS=[] -> bo qua. Dat SEEDS=[1,2] de co con so sigma.")

## 11. Inference — dùng model thật

Hàm `sql()` dưới đây là thứ bạn đem đi deploy.

In [ ]:
ft.eval()
@torch.no_grad()
def sql(schema, question, model=ft, max_new_tokens=96):
    p = build_prompt({"context":schema, "question":question})
    b = tok(p, return_tensors="pt", add_special_tokens=False).to("cuda")
    t0=time.time()
    g = model.generate(**b, max_new_tokens=max_new_tokens, do_sample=False,
                       pad_token_id=tok.pad_token_id)
    out = tok.decode(g[0][b["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return out, time.time()-t0

tests = [
 ("CREATE TABLE nhan_vien (ho_ten VARCHAR, luong INTEGER, phong_ban VARCHAR)",
  "Luong trung binh cua phong ban Ky thuat la bao nhieu?"),
 ("CREATE TABLE don_hang (id INTEGER, khach VARCHAR, tong_tien INTEGER, ngay DATE)",
  "Dem so don hang co tong tien tren 500000"),
 ("CREATE TABLE sinh_vien (ten VARCHAR, diem FLOAT, lop VARCHAR)",
  "Liet ke ten sinh vien lop 12A co diem cao nhat"),
]
for s,q in tests:
    out,dt = sql(s,q)
    print(f"Hoi : {q}\nSQL : {out}\n      ({dt*1000:.0f} ms)\n")

In [ ]:
# Do throughput - con so can biet truoc khi deploy
lat=[sql(*tests[0])[1] for _ in range(5)]
print(f"latency: trung binh {1000*np.mean(lat):.0f} ms | min {1000*min(lat):.0f} | max {1000*max(lat):.0f}")
print(f"VRAM dang dung: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

## 12. Merge + xuất model để deploy

LoRA adapter cần model gốc mới chạy. `merge_and_unload()` gộp adapter vào weight → được model
độc lập, deploy đơn giản hơn (và là bước đầu để convert sang GGUF cho Ollama).

In [ ]:
# SAVE_MERGED=False mac dinh: `kaggle kernels output` tai TOAN BO /kaggle/working,
# khong chon loc duoc. Luu model merge (~1.2GB) lam moi lan tai ket qua rat cham.
# Chi bat khi that su can file model de deploy.
SAVE_MERGED = False
MERGED="/kaggle/working/qwen3-0.6b-sql-merged"
if SAVE_MERGED:
    merged = PeftModel.from_pretrained(
        AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE), OUT).merge_and_unload()
    merged.save_pretrained(MERGED); tok.save_pretrained(MERGED)
    print("da luu model merge:", MERGED)
else:
    print("bo qua merge (SAVE_MERGED=False) -> tai ket qua nhanh. Adapter van duoc luu.")

metrics = {"model":MODEL, "n_train":N_TRAIN, "lora_r":16, "lr":2e-4,
           "train_seconds":round(train_secs,1), "dtype":str(DTYPE),
           "gpu":torch.cuda.get_device_name(0),
           "acc_base":round(acc_b,2), "acc_lora":round(acc_f,2),
           "hit_base":h_base, "hit_lora":h_ft, "n_eval":len(ev),
           "final_train_loss":tr[-1][1], "final_val_loss":(va[-1][1] if va else None),
           "acc_seeds":accs, "acc_mean":mu, "acc_std":sd_,
           "bf16_reported_by_torch":torch.cuda.is_bf16_supported(), "compute_cap":list(cap)}
json.dump(metrics, open("/kaggle/working/metrics.json","w"), indent=2)
print(json.dumps(metrics, indent=2))
print("\nAdapter (nho, ~40MB):", OUT)
print("Model da merge      :", MERGED)
print("\nBuoc tiep: convert GGUF -> Ollama (xem smartkids-finetune cua ban)")

## 13. Thí nghiệm — vừa chạy vừa học

Mỗi thí nghiệm có **kết quả kỳ vọng**. Ra khác nghĩa là bạn học được điều mới (hoặc có bug).
Chạy lại ô 7→10 sau mỗi lần sửa để thấy biểu đồ đổi.

| # | Sửa | Kỳ vọng |
|---|---|---|
| 1 | `N_TRAIN=20000` | Điểm tăng **không tuyến tính**. Đo: 5x thời gian được mấy %? |
| 2 | `learning_rate=2e-2` | Loss **nổ NaN**. Thấy tận mắt lr quá cao. |
| 3 | `learning_rate=2e-6` | Loss gần như không giảm, điểm ~0%. Underfit. |
| 4 | `labels=(p+a)[:MAX_LEN]` (bỏ masking ở ô 3) | Loss vẫn giảm đẹp **nhưng điểm eval tệ hơn**. Lý do phải mask. |
| 5 | `r=4`, rồi `r=64` | r=4 có thể đã đủ. r=64 nặng hơn mà không tốt hơn. |
| 6 | `num_train_epochs=3` | **Biểu đồ ô 8 sẽ cho thấy val loss tăng** trong khi train loss giảm = overfit. |
| 7 | `target_modules=["q_proj","v_proj"]` | Kém hơn rõ so với phủ hết 7 linear. |
| 8 | `MODEL="Qwen/Qwen3-1.7B"` | Tốt hơn. OOM → `batch=2` + `gradient_checkpointing=True`. |

### Tiếp theo
- **Track 02 — DPO:** dạy theo *sở thích* thay vì nhãn đúng/sai
- **Track 04 — OCR:** ảnh hóa đơn → JSON. Đường tới **OmniDocBench**, sân duy nhất model nhỏ
  đứng trên GPT-5.2 (PaddleOCR-VL 0.9B đạt 96.34 vs GPT-5.2 86.59).